# BoFM Ensemble Parse Pipeline

Multi-parser ensemble for Universal Dependencies parsing of the Book of Mormon corpus. Outputs CoNLL-U per parser to Google Drive for downstream agreement-measurement and adjudication.

**Parsers:**
- stanza (UD-trained, multilingual)
- spaCy en_core_web_trf (transformer)
- Trankit (adaptive transformer)
- UDPipe (neural UD)

**Pattern:** mirrors `samuel_pipeline.ipynb` (Drive persistence + git clone)

**Workflow:**
1. Cell 1-2: setup (install parsers, mount Drive, clone repo)
2. Cell 3: helper functions (prose extraction + per-parser adapters)
3. Cell 4: dry-run on Alma 30 (verify pipeline)
4. Cell 5: full ensemble run on Alma 30 (Phase 0 reconnaissance output)
5. Cell 6: full-corpus run (Phase 1 — only when Phase 0 is confirmed green)

In [ ]:
# Cell 1: Install parsers + Mount Google Drive
# This takes ~5 min on first run; subsequent runs reuse Colab cache

!pip install -q stanza spacy spacy_conll trankit ufal.udpipe
!python -m spacy download en_core_web_trf

from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted!')

In [ ]:
# Cell 2: Clone repo + setup paths
# IMPORTANT: Push your local commits BEFORE running this cell!

!git clone https://github.com/bibleman-stan/readers-bofm.git
%cd readers-bofm

import os
from pathlib import Path

DRIVE_BASE = '/content/drive/MyDrive/bom-reader-parses'
DRIVE_PARSES = f'{DRIVE_BASE}/ensemble'
os.makedirs(DRIVE_PARSES, exist_ok=True)

# Per-parser output directories
for parser_name in ['stanza', 'spacy', 'trankit', 'udpipe']:
    os.makedirs(f'{DRIVE_PARSES}/{parser_name}', exist_ok=True)

print('Setup complete.')
print(f'  Repo: {os.getcwd()}')
print(f'  Drive output: {DRIVE_PARSES}')

In [ ]:
# Cell 3: Helper functions — prose extraction + per-parser adapters

import re
from pathlib import Path

CORPUS_DIR = Path('data/text-files/v2-mine')

# Map book_id → source filename and chapter line ranges
BOOK_FILES = {
    '1nephi': '01-1_nephi-2020-sb-v2.txt',
    '2nephi': '02-2_nephi-2020-sb-v2.txt',
    'jacob': '03-jacob-2020-sb-v2.txt',
    'enos': '04-enos-2020-sb-v2.txt',
    'jarom': '05-jarom-2020-sb-v2.txt',
    'omni': '06-omni-2020-sb-v2.txt',
    'words-of-mormon': '07-words_of_mormon-2020-sb-v2.txt',
    'mosiah': '08-mosiah-2020-sb-v2.txt',
    'alma': '09-alma-2020-sb-v2.txt',
    'helaman': '10-helaman-2020-sb-v2.txt',
    '3nephi': '11-3_nephi-2020-sb-v2.txt',
    '4nephi': '12-4_nephi-2020-sb-v2.txt',
    'mormon': '13-mormon-2020-sb-v2.txt',
    'ether': '14-ether-2020-sb-v2.txt',
    'moroni': '15-moroni-2020-sb-v2.txt',
}

VERSE_REF_RE = re.compile(r'^\d+:\d+\s*$')


def extract_prose(book_id, chapter_num=None):
    """Extract prose text for a book or single chapter.
    
    Returns: (text, atu_line_count) where text is space-joined ATU lines
    suitable for parser input, and atu_line_count is the source line count.
    """
    fp = CORPUS_DIR / BOOK_FILES[book_id]
    with open(fp, encoding='utf-8') as f:
        lines = f.readlines()
    
    in_chapter = chapter_num is None  # if no chapter specified, take all
    target_marker = f'{chapter_num}:' if chapter_num else None
    next_marker = f'{chapter_num + 1}:1' if chapter_num else None
    
    prose = []
    for line in lines:
        s = line.strip()
        if not s:
            continue
        if VERSE_REF_RE.match(s):
            if chapter_num is not None:
                if s.startswith(f'{chapter_num}:'):
                    in_chapter = True
                    continue
                elif s == next_marker or (in_chapter and s.startswith(f'{chapter_num + 1}:')):
                    in_chapter = False
                    break
            continue
        if in_chapter:
            prose.append(s)
    
    return ' '.join(prose), len(prose)


# ============================================================
# Per-parser adapters — each takes text, returns CoNLL-U string
# ============================================================

def parse_stanza(text):
    import stanza
    if not hasattr(parse_stanza, 'nlp'):
        parse_stanza.nlp = stanza.Pipeline(
            'en', processors='tokenize,pos,lemma,depparse',
            verbose=False, use_gpu=True,
        )
    doc = parse_stanza.nlp(text)
    from stanza.utils.conll import CoNLL
    return '\n'.join(['\t'.join(map(str, tok)) for sent in CoNLL.convert_dict(doc.to_dict()) for tok in sent] + [''])


def parse_spacy(text):
    import spacy
    from spacy_conll import init_parser
    if not hasattr(parse_spacy, 'nlp'):
        parse_spacy.nlp = init_parser(
            'en_core_web_trf',
            'spacy',
            include_headers=True,
        )
    doc = parse_spacy.nlp(text)
    return doc._.conll_str


def parse_trankit(text):
    from trankit import Pipeline
    if not hasattr(parse_trankit, 'nlp'):
        parse_trankit.nlp = Pipeline('english', gpu=True)
    out = parse_trankit.nlp(text)
    # Trankit returns dict; convert to CoNLL-U manually
    lines = []
    for sent in out['sentences']:
        for tok in sent['tokens']:
            row = [
                str(tok['id']),
                tok['text'],
                tok.get('lemma', '_'),
                tok.get('upos', '_'),
                tok.get('xpos', '_'),
                tok.get('feats', '_'),
                str(tok.get('head', 0)),
                tok.get('deprel', '_'),
                '_',
                '_',
            ]
            lines.append('\t'.join(row))
        lines.append('')
    return '\n'.join(lines)


def parse_udpipe(text):
    from ufal.udpipe import Model, Pipeline as UDPipeline, ProcessingError
    if not hasattr(parse_udpipe, 'pipeline'):
        # Download UD English model if not cached
        model_path = '/content/english-ewt-ud-2.5-191206.udpipe'
        if not os.path.exists(model_path):
            import urllib.request
            url = 'https://lindat.mff.cuni.cz/repository/xmlui/bitstream/handle/11234/1-3131/english-ewt-ud-2.5-191206.udpipe'
            urllib.request.urlretrieve(url, model_path)
        model = Model.load(model_path)
        parse_udpipe.pipeline = UDPipeline(
            model, 'tokenize', UDPipeline.DEFAULT, UDPipeline.DEFAULT, 'conllu'
        )
    error = ProcessingError()
    return parse_udpipe.pipeline.process(text, error)


PARSERS = {
    'stanza': parse_stanza,
    'spacy': parse_spacy,
    'trankit': parse_trankit,
    'udpipe': parse_udpipe,
}

print('Helpers loaded.')
print(f'  Parsers: {list(PARSERS.keys())}')
print(f'  Books: {len(BOOK_FILES)}')

In [ ]:
# Cell 4: Dry run — extract Alma 30 prose, verify each parser loads + parses

text, line_count = extract_prose('alma', chapter_num=30)
print(f'Alma 30: {line_count} ATU lines, {len(text)} chars')
print(f'First 200 chars: {text[:200]}...')
print()

# Verify each parser can produce CoNLL-U on a small sample
sample = text[:500]
for name, fn in PARSERS.items():
    print(f'Testing {name}...')
    try:
        out = fn(sample)
        first_lines = '\n'.join(out.split('\n')[:5])
        print(f'  OK. First 5 lines:\n{first_lines}\n')
    except Exception as e:
        print(f'  FAILED: {e}\n')

In [ ]:
# Cell 5: Full ensemble run — Alma 30 with all 4 parsers, save to Drive
# Phase 0 reconnaissance output. Allows local agreement_measure.py.

import time

text, line_count = extract_prose('alma', chapter_num=30)
print(f'Parsing Alma 30 ({line_count} ATU lines, {len(text)} chars) with 4 parsers...')
print()

for name, fn in PARSERS.items():
    print(f'[{name}] running...')
    t0 = time.time()
    conllu = fn(text)
    elapsed = time.time() - t0
    
    out_path = f'{DRIVE_PARSES}/{name}/alma-30.conllu'
    with open(out_path, 'w', encoding='utf-8') as f:
        f.write(conllu)
    
    n_tokens = sum(1 for line in conllu.split('\n')
                   if line and not line.startswith('#') and '\t' in line)
    print(f'  saved {n_tokens} tokens to {out_path} ({elapsed:.1f}s)')

print()
print('Phase 0 ensemble complete.')
print(f'Outputs: {DRIVE_PARSES}/{{stanza,spacy,trankit,udpipe}}/alma-30.conllu')
print()
print('Next: download these locally and run validators/parsing/agreement_measure.py')

In [ ]:
# Cell 6: FULL CORPUS RUN — only run after Phase 0 agreement report is reviewed
# All 15 books × 4 parsers. Estimated ~30-60 min on T4 GPU.

RUN_FULL_CORPUS = False  # ← change to True only after Phase 0 review

if not RUN_FULL_CORPUS:
    print('RUN_FULL_CORPUS is False. Set to True to proceed.')
else:
    import time
    for book_id in BOOK_FILES.keys():
        text, line_count = extract_prose(book_id)
        print(f'\n[{book_id}] {line_count} ATU lines, {len(text)} chars')
        for name, fn in PARSERS.items():
            t0 = time.time()
            try:
                conllu = fn(text)
                out_path = f'{DRIVE_PARSES}/{name}/{book_id}.conllu'
                with open(out_path, 'w', encoding='utf-8') as f:
                    f.write(conllu)
                elapsed = time.time() - t0
                n_tokens = sum(1 for line in conllu.split('\n')
                               if line and not line.startswith('#') and '\t' in line)
                print(f'  [{name}] {n_tokens} tokens ({elapsed:.1f}s)')
            except Exception as e:
                print(f'  [{name}] FAILED: {e}')
    print('\nFull corpus parse complete.')